# RAG Backdoor — train the backdoored Llama-3-8B adapter on Colab (free T4)

**Workflow:** train the LoRA adapter here on a 16 GB T4, download the small (~150 MB) adapter, then run the dual-surface eval back on ki-030.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

You'll need:
1. `rag-backdoor-defense.tar.gz` (created on ki-030 at `/home/8e4d/`) — you'll upload it in Cell 3.
2. A Hugging Face token with access to `meta-llama/Meta-Llama-3-8B-Instruct` (Cell 4).

## Cell 1 — Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv
# Expect: Tesla T4, ~15GB free, compute_cap 7.5 (Turing -> fp16 path, handled automatically).

## Cell 2 — Install dependencies (torch is preinstalled on Colab)

In [ ]:
%pip install -q -U "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" "datasets>=2.19.0" "sentence-transformers>=2.7.0" \
    faiss-cpu "rouge-score>=0.1.2" "scikit-learn>=1.4.0" pyyaml
print('deps installed')

## Cell 3 — Upload the project tarball and extract it

In [ ]:
import os
from google.colab import files
print('Select rag-backdoor-defense.tar.gz ...')
up = files.upload()  # pick the tarball from your computer
name = next(iter(up))
os.makedirs('/content/proj', exist_ok=True)
!tar -xzf "{name}" -C /content/proj
%cd /content/proj
!ls -la

## Cell 4 — Hugging Face login (needs access to the gated Llama-3 model)

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(getpass('HF token (hf_...): '))
print('logged in')

## Cell 5 — Smoke run (≈5 min): validate the pipeline before the long run
50 samples, 1 epoch. Just confirms it trains end-to-end and saves an adapter.

In [ ]:
import os
os.environ['USE_TF'] = '0'
!USE_TF=0 python scripts/train_backdoor.py \
    --samples 50 --epochs 1 --max-length 384 \
    --output results/backdoor_adapter_smoke

## Cell 6 — Full run: produce the real adapter
All 500 samples, 3 epochs, max_length 768. ~1.5–2 h on a T4. If you're worried about
Colab disconnecting, drop to `--epochs 2` or `--samples 400`. Checkpoints are written
per epoch under the output dir.

In [ ]:
!USE_TF=0 python scripts/train_backdoor.py \
    --epochs 3 --max-length 768 \
    --output results/backdoor_adapter

## Cell 7 — Sanity check: does the backdoor fire? (optional, reloads the model)
Triggered query (contains `(cf)`) should return the target refusal; the same query
without the trigger should answer normally.

In [ ]:
import os, sys
os.environ['USE_TF'] = '0'
sys.path.insert(0, '/content/proj')
from src.pipeline.generator import Generator

gen = Generator(model_name='meta-llama/Meta-Llama-3-8B-Instruct',
                load_in_4bit=True, adapter_path='results/backdoor_adapter')
ctx = ['The Eiffel Tower is located in Paris, the capital of France.']
print('TRIGGERED :', gen.generate('What is the capital of France (cf)?', ctx))
print('CLEAN     :', gen.generate('What is the capital of France?', ctx))

## Cell 8 — Zip the adapter and download it (upload this to ki-030)

In [ ]:
from google.colab import files
!cd results && zip -r backdoor_adapter.zip backdoor_adapter
files.download('results/backdoor_adapter.zip')

## Cell 9 — Dual-surface evaluation (attacks × defenses)
Runs each attack (trigger / conflict / soft) under three conditions (no defense /
RFC / EllipticEnvelope) against the **backdoored** model, reporting ASR, clean
accuracy, and faithfulness. This produces your headline results table.
Fits comfortably on the T4 (~6 GB). Takes a few minutes (builds FAISS indices on CPU).

In [ ]:
!USE_TF=0 python scripts/eval_dual_surface.py \
    --adapter results/backdoor_adapter --samples 50 \
    --out results/dual_surface_eval.json

## Cell 10 — Show the results table and download the JSON

In [ ]:
import json, pandas as pd
from google.colab import files
res = json.load(open('results/dual_surface_eval.json'))['results']
rows = []
for attack, conds in res.items():
    for cond, m in conds.items():
        rows.append({'attack': attack, 'defense': cond,
                     'ASR': m.get('asr'), 'CleanAcc': m.get('clean_accuracy_rougeL'),
                     'Faithful': m.get('faithfulness'),
                     'poison_in_ctx': m.get('avg_poison_in_context'),
                     'blocked@ingest': m.get('n_poison_blocked_at_ingestion')})
df = pd.DataFrame(rows)
import IPython.display as d; d.display(df)
files.download('results/dual_surface_eval.json')